# FULL-FINAL — Utah FORGE 56-32 Geothermal Drilling Sensor Clustering with K-Means

**Title (IEEE):** Impact of Feature Scaling and Temporal Window Aggregation on Multivariate Geothermal Drilling Sensor Clustering Using K-Means

**Final runnable notebook.** Supersedes `utah_forge_56_32_geothermal_drilling_sensor_clust-fathan.ipynb` (kept as pre-final archive). Single run on CPU (~30 min), RAM-safe (chunked reads, sampled K-Means).
Locked: SEED=42, 7 modeled sensors (Diff Press OFF), ROP winsorized via ROP>0 IQR, strided full-span samples, gap-aware 60s/30s windows.

## Step-by-step map (use this table in paper Bab II as pipeline figure)

| Step | Goal | Input → Output | Paper section |
|---|---|---|---|
| 0. Setup | Resolve input (Kaggle/local/kagglehub), seed=42, helpers | Dataset `utah-forge-56-32-1sec` | Reproducibility note |
| 1.0 Taxonomy | Formal sensor taxonomy, units, groups, selection | Sensor list → `table0_sensor_taxonomy.csv` | Bab II-A |
| 1. EDA | Descriptives, scale audit, correlation, boxplot, sentinel audit | Strided 200k → `table1_descriptive.csv`, `fig1_correlation.png`, `fig2_boxplots.png` | Bab II-A |
| 2. Scaling | Euclidean bias; Standard vs Robust (+stability, S, timing, gate) | 20k sample, k=4 → `table2_scaler_compare.csv`, `fig3_scaling_hist.png` | Bab II-B |
| 3. Windowing | Temporal dependency (Keogh & Lin 2005) | Contiguous 30k → 60s/30s mean+std+delta (21-D) → `table3_temporal_compare.csv`, `fig4_temporal_trace.png` | Bab II-C |
| 4. Clustering | Elbow k=2..8 x 4 scenarios, Silhouette + DBI, gate, centroids → regimes | Points strided 60k + full-file windows → `table4_elbow_metrics.csv`, `fig5_elbow_silhouette.png`, `table5_centroids.csv`, `table6_cluster_share.csv`, `fig6_pca_clusters.png` | Bab III |
| 5. Export | Recap + MISSING-check + download for IEEE paper | All tables+figures | Appendix + repo link |

> **Setup:** Kaggle → Add Input `utah-forge-56-32-1sec` (or auto-download via kagglehub); Colab → `pip install kagglehub` + creds. CPU only, captions English-ready.

#### Cell: Setup — imports, locked constants, helpers, input resolver
**What:** libraries, the 8-candidate list (`SENSOR_8`) + 7 modeled (`SENSORS`), sample sizes, `SEED=42`, `rop_hi()` (never-0 winsor cap), `physicality_gate()`, composite `S`, and `resolve_input()` (Kaggle → local → kagglehub auto-download) with a Colab-aware `OUT`.
**Why:** one seed, one path rule and shared helpers keep every step reproducible and mutually comparable.
**Read this:** printed `INPUT` path, `OUTPUT` folder (absolute), and sensor count (must be 7).

In [ ]:
from pathlib import Path
import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

SENSOR_8 = ['Rate Of Penetration (ft_per_hr)','Weight on Bit (klbs)','Rotary RPM (RPM)','Standpipe Pressure (psi)','Rotary Torque (kft_lb)','Hook Load (klbs)','Total Pump Output (gal_per_min)','Differential Pressure (psi)']
ROP = 'Rate Of Penetration (ft_per_hr)'
DIFF = 'Differential Pressure (psi)'
SENTINEL = -999.25
CSV_NAME = '56-32 1sec data 27029986.csv'
SEED = 42
N_EDA, N_SCALE, N_WIN_SRC, N_FINAL = 200000, 20000, 30000, 60000
W, STEP = 60, 30
STAB_SEEDS = [42, 7, 11, 23, 99]
USE_DIFF_PRESS = False  # locked default: 7 modeled sensors (True = 8-sensor sensitivity)
SENSORS = SENSOR_8 if USE_DIFF_PRESS else [c for c in SENSOR_8 if c != DIFF]

def rop_hi(frame, col=ROP):
    pos = frame[col][frame[col] > 0]  # drilling-only distribution; idle zeros and -999.25 excluded
    base = pos if len(pos) >= 100 else frame[col]
    q1, q3 = base.quantile(0.25), base.quantile(0.75)
    hi = float(q3 + 1.5*(q3-q1))
    if not np.isfinite(hi) or hi <= 0:
        hi = float(frame[col].quantile(0.99))  # fallback for zero-heavy samples
    return hi

def norm_sil(s):
    return float((s + 1.0) / 2.0)

def norm_dbi(d):
    return float(1.0 / (1.0 + d))

def composite(sil, dbi, w_sil=0.5):
    return float(w_sil * norm_sil(sil) + (1.0 - w_sil) * norm_dbi(dbi))

def physicality_gate(cent):
    fails = []
    use = [c for c in cent.columns if c != DIFF]
    if (cent[use].to_numpy() < 0).any():
        fails.append('negative-sensor-value')
    rpm = cent['Rotary RPM (RPM)']; rop = cent[ROP]
    spp = cent['Standpipe Pressure (psi)']; wob = cent['Weight on Bit (klbs)']
    if ((rop > 50) & ((spp < 500) | (rpm < 10))).any():
        fails.append('drilling-without-pump-or-rotation')
    if ((spp < 100) & ((rop > 50) | (wob > 50))).any():
        fails.append('load-without-pump')
    return (len(fails) == 0), fails

def _kagglehub_csv(csv=CSV_NAME):
    try:
        import kagglehub
    except ImportError:
        print('kagglehub not installed: pip install kagglehub[pandas-datasets]')
        return None
    try:
        d = Path(kagglehub.dataset_download('faruqmahdison/utah-datmin'))
    except Exception as e:
        print('kagglehub download failed (needs Kaggle creds on Colab/local):', e)
        return None
    hits = list(d.rglob(csv))
    if hits:
        return hits[0]
    csvs = [p for p in d.rglob('*.csv') if p.is_file()]
    return max(csvs, key=lambda p: p.stat().st_size) if csvs else None

def resolve_input(csv=CSV_NAME):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    print('Local CSV not found — trying kagglehub: faruqmahdison/utah-datmin ...')
    hit = _kagglehub_csv(csv)
    if hit is not None:
        return hit
    raise FileNotFoundError('Dataset CSV not found. Fix A (Kaggle): right panel Add Input -> add dataset faruqmahdison/utah-datmin -> re-run. Fix B (anywhere): pip install kagglehub + Kaggle API creds, then re-run.')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if Path('/kaggle/output').exists():
    OUT = Path('/kaggle/output')
elif IN_COLAB:
    OUT = Path('fathan_clustering_outputs')  # Colab: local downloadable folder
else:
    OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)
print('INPUT:', resolve_input())
print('OUTPUT:', OUT.resolve())
print('Modeling sensors:', len(SENSORS), '(Diff Press OFF locked)' if not USE_DIFF_PRESS else '(sensitivity: 8)')

#### Cell: Sensor Taxonomy (formal drilling specification) → `table0_sensor_taxonomy.csv`
**What:** formal taxonomy for all logging channels: units, groups, physical roles, and selection status (7 modeled / Diff-Press excluded-OFF / Gas+Gamma dead / depth cols = metadata).
**Why:** the paper Bab II-A must open with 22 → 8 candidates → 7 modeled, each exclusion evidenced.
**Read this:** Diff-Press status must read Excluded-OFF (buggy derived channel), never Modeled.

In [ ]:
import pandas as pd

# Formal Sensor Taxonomy (Utah FORGE 56-32) — statuses match locked decisions
# (Diff Press default OFF: buggy derived channel; Gas/Gamma dead; depth cols = metadata)
sensor_taxonomy_dict = {
    'Rate Of Penetration (ft_per_hr)': {'Abbreviation': 'ROP', 'Unit': 'ft/hr', 'Measurement Group': 'Drilling Performance (Kinematics)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Axial speed of bit penetration through rock.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Weight on Bit (klbs)': {'Abbreviation': 'WOB', 'Unit': 'klbs', 'Measurement Group': 'Mechanical (Axial Load)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Axial load applied onto the drill bit.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Rotary RPM (RPM)': {'Abbreviation': 'RPM', 'Unit': 'rpm', 'Measurement Group': 'Mechanical (Rotational Kinematics)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Rotational frequency of the drill string.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Standpipe Pressure (psi)': {'Abbreviation': 'SPP', 'Unit': 'psi', 'Measurement Group': 'Hydraulic (Fluid Circulation)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Mud pressure at the rig standpipe.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Rotary Torque (kft_lb)': {'Abbreviation': 'Torque', 'Unit': 'kft_lb', 'Measurement Group': 'Mechanical (Rotational Torque)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Moment required to rotate the string against rock resistance.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Hook Load (klbs)': {'Abbreviation': 'HKLD', 'Unit': 'klbs', 'Measurement Group': 'Mechanical (Axial Tensile Load)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Pull force on the traveling block; basis for WOB.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Total Pump Output (gal_per_min)': {'Abbreviation': 'TPO', 'Unit': 'gal_per_min', 'Measurement Group': 'Hydraulic (Volumetric Flow Rate)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Mud flow rate circulated by the pumps.', 'Selection Status': 'Primary Feature (Modeled)'},
    'Differential Pressure (psi)': {'Abbreviation': 'Diff Press', 'Unit': 'psi', 'Measurement Group': 'Hydraulic (Downhole Motor Load)', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Derived pressure difference (Pason software channel).', 'Selection Status': 'Excluded by default (buggy derivation, mean negative; 8-sensor sensitivity only)'},
    'Pason Gas (percent)': {'Abbreviation': 'Gas', 'Unit': '%', 'Measurement Group': 'Geochemical / Mud Logging', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Hydrocarbon gas fraction in mud.', 'Selection Status': 'Excluded (Sentinel ~100% -999.25, dead sensor)'},
    'Gamma (api)': {'Abbreviation': 'GR', 'Unit': 'api', 'Measurement Group': 'Petrophysical / MWD-LWD', 'Data Type': 'Continuous (Ratio)', 'Physical Meaning': 'Natural gamma radiation of formation.', 'Selection Status': 'Excluded (Sentinel ~100% -999.25, dead sensor)'},
}
taxonomy_rows = []
for col_name, info in sensor_taxonomy_dict.items():
    taxonomy_rows.append({'Sensor / Dataset Column': col_name, **info})
sensor_taxonomy = pd.DataFrame(taxonomy_rows)
try:
    display(sensor_taxonomy)
except NameError:
    print(sensor_taxonomy.to_string(index=False))
sensor_taxonomy.to_csv(OUT/'table0_sensor_taxonomy.csv', index=False)
print('saved table0_sensor_taxonomy.csv')

## Step 1 — EDA (Data Understanding)
**Why:** (a) quantify scale disparity (psi ~thousands vs torque ~tens → Euclidean bias), (b) confirm `Pason Gas`/`Gamma` are inactive (`-999.25`), (c) correlation + outliers for sensor selection (need ≥5 continuous numeric, unlabeled). Strided full-span 200k sample doubles as cross-check of Orang 2's Colab 200k descriptives.

#### Cell: Load EDA sample (strided full-span 200k)
**What:** counts total rows, then reads every k-th row across the whole file (time-sorted), downcasts to `float32`, builds a `datetime` index.
**Why:** head-only samples bias to February drilling; strided sampling covers Feb→Mar including outage bursts, staying comparable with the locked 200k Colab basis.
**Read this:** printed stride, row count, and time span (must cover the full month).

In [ ]:
path = resolve_input()
usecols = ['YYYY/MM/DD','HH:MM:SS'] + SENSOR_8
n_e = 0
for _c in pd.read_csv(path, usecols=[SENSOR_8[0]], chunksize=500000, low_memory=True):
    n_e += len(_c)
step_e = max(1, n_e // N_EDA)
eda = pd.read_csv(path, usecols=usecols, skiprows=lambda i: i % step_e != 0, low_memory=True)
print('EDA strided sample: every', step_e, 'th row of', n_e, '->', len(eda), 'rows')
for c in SENSOR_8: eda[c] = pd.to_numeric(eda[c], errors='coerce').astype('float32')
eda['datetime'] = pd.to_datetime(eda['YYYY/MM/DD']+' '+eda['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
eda = eda.sort_values('datetime').reset_index(drop=True)
print(eda.shape, eda['datetime'].min(), '->', eda['datetime'].max())
display(eda.head(2))

#### Cell: Descriptive statistics → `table1_descriptive.csv`
**What:** mean, std, min/max, quartiles, skewness, range per candidate sensor (sentinel excluded first) on the strided sample.
**Why:** the scale-disparity proof (psi in thousands vs torque in tens) that forces scaling in Step 2.
**Read this:** compare std/range of Standpipe Pressure vs Rotary Torque — that gap IS the scale-disparity disease.

In [ ]:
desc = eda[SENSOR_8].replace(-999.25, np.nan).describe(percentiles=[.25,.5,.75]).T
desc['skew'] = eda[SENSOR_8].replace(-999.25, np.nan).skew(numeric_only=True)
desc['range'] = desc['max'] - desc['min']
display(desc.round(2))
desc.round(2).to_csv(OUT/'table1_descriptive.csv')
print('saved table1_descriptive.csv — reconcile means/std with Orang 2 Colab 200k (cross-check, not conflict)')

#### Cell: Missing-code audit (`-999.25`)
**What:** sentinel rate per channel on the strided sample (representative) + the 20k Gas/Gamma probe.
**Why:** `-999.25` means a dead channel, not a measurement; Gas/Gamma at ~100% justifies permanent exclusion.
**Read this:** SPP/Diff-Press ~1.17% (systemic outage bursts) vs ~0% elsewhere → dropna-subset loses ~1% rows.

In [ ]:
chk = pd.read_csv(path, usecols=['Pason Gas (percent)','Gamma (api)'], nrows=20000)
print('Pason Gas == -999.25:', float((chk['Pason Gas (percent)']==-999.25).mean()))
print('Gamma == -999.25:', float((chk['Gamma (api)']==-999.25).mean()))
print('Decision: DROP both (inactive sensors). Keep the 8-candidate set.')
strate = {c: round(100*float((eda[c]==-999.25).mean()), 3) for c in SENSOR_8}
print('sentinel % on strided sample:', strate)

#### Cell: Correlation + boxplots → `fig1`, `fig2`
**What:** 8×8 correlation heatmap over the 8 CANDIDATE sensors (Diff-Press included as excluded-channel evidence) + one boxplot per sensor (1st–99th pct clipped for readability).
**Why:** correlation reveals coupled subsystems (hydraulic/rotational); boxplots reveal the ROP drawworks spike. Modeling itself uses 7 sensors (see setup).
**Read this:** SPP–Pump ≈0.9 (kept both: different physics, not a drop reason) and ROP whiskers = artefacts to winsorize, not delete.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(eda[SENSOR_8].replace(-999.25, np.nan).corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Fig 1. Correlation Matrix — 8 Geothermal Drilling Sensor Candidates')
plt.tight_layout(); plt.savefig(OUT/'fig1_correlation.png', dpi=150); plt.show()

fig, ax = plt.subplots(2,4, figsize=(14,6))
for a, c in zip(ax.ravel(), SENSOR_8):
    v = eda[c].replace(-999.25, np.nan).dropna().to_numpy()
    lo, hi = np.percentile(v,[1,99])
    a.boxplot(v[(v>=lo)&(v<=hi)], vert=True); a.set_title(c.split(' (')[0], fontsize=9)
plt.suptitle('Fig 2. Boxplots (1st–99th pct clipped) — Outlier Detection')
plt.tight_layout(); plt.savefig(OUT/'fig2_boxplots.png', dpi=150); plt.show()

## Step 2 — Scaling (StandardScaler vs RobustScaler + stability, S, timing, gate)
**Why:** K-Means uses Euclidean distance. Without scaling, `Standpipe Pressure` (0–3500 psi) dominates `Torque` (1–25) and `WOB` (0–50). `RobustScaler` (median/IQR) should resist the ROP drawworks spike; this demo proves it at k=4 on 7 modeled sensors.

#### Cell: Scaler comparison → `table2_scaler_compare.csv`
**What:** K-Means (k=4) three ways — unscaled, `StandardScaler`, `RobustScaler` — reporting Silhouette, DBI, composite S, seed-stability over 5 seeds, convergence, and the centroid physics gate.
**Why:** mini-P1: the scaler decision with numbers, not claims. Winner = highest S among gate-passed.
**Read this:** Robust must lead S; unscaled may look good on Silhouette but lives on the zero-mass — check its gate.

In [ ]:
s = pd.read_csv(path, usecols=SENSORS, nrows=N_SCALE*2, low_memory=True)
for c in SENSORS: s[c]=pd.to_numeric(s[c], errors='coerce').astype('float32')
s = s.replace(SENTINEL, np.nan).dropna().reset_index(drop=True)
ROP_HI = rop_hi(s)
assert ROP_HI > 0, 'degenerate sample: ROP all-zero'
s[ROP] = s[ROP].clip(upper=ROP_HI)
print('ROP winsorized at HI =', round(ROP_HI, 2))
sdf = s.sample(N_SCALE, random_state=SEED).reset_index(drop=True)
s = sdf.to_numpy()
print(pd.DataFrame({'sensor':SENSORS,'std':s.std(axis=0)}).sort_values('std',ascending=False).to_string(index=False))
res2={}
for name, sc in [('unscaled',None),('standard',StandardScaler()),('robust',RobustScaler())]:
    X = s if sc is None else sc.fit_transform(s)
    sil_seed = []
    for sd in STAB_SEEDS:
        lab = KMeans(n_clusters=4, n_init=10, random_state=sd).fit_predict(X)
        sil_seed.append(float(silhouette_score(X,lab)))
    t0f=time.time(); km=KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(X); tfit=round(time.time()-t0f,2)
    lab = km.labels_
    sil=float(silhouette_score(X,lab)); dbi=float(davies_bouldin_score(X,lab))
    cent = sc.inverse_transform(km.cluster_centers_) if sc is not None else km.cluster_centers_
    passed, fails = physicality_gate(pd.DataFrame(cent, columns=SENSORS))
    S=composite(sil,dbi)
    res2[name]={'silhouette':round(sil,4),'sil_std_over_seeds':round(float(np.std(sil_seed)),6),'dbi':round(dbi,4),'S4848':round(S,4),'S4060':round(composite(sil,dbi,0.4),4),'S6040':round(composite(sil,dbi,0.6),4),'n_iter':int(km.n_iter_),'fit_seconds':tfit,'gate_passed':passed,'gate_fails':';'.join(fails) if fails else '-'}
    print(name, {k:(round(v,4) if isinstance(v,float) else v) for k,v in res2[name].items()})
pd.DataFrame(res2).T.to_csv(OUT/'table2_scaler_compare.csv')
print('Winner = highest S among gate-passed; S=0.50*Sil_norm+0.50*(1-DBI_norm).')

#### Cell: Distribution plot → `fig3_scaling_hist.png`
**What:** overlaid log-scale histograms of raw vs Standard-scaled vs Robust-scaled values.
**Why:** visual proof that scaling puts all sensors on a comparable footing.
**Read this:** Robust histogram compact without long tails = spike tamed.

In [ ]:
xs, ss, rs = s, StandardScaler().fit_transform(s), RobustScaler().fit_transform(s)
fig, ax = plt.subplots(1,3, figsize=(12,3), sharey=True)
for a, X, t in zip(ax,[xs,ss,rs],['Raw','StandardScaler','RobustScaler']):
    a.hist(X.ravel(), bins=80); a.set_title(t); a.set_yscale('log')
plt.suptitle('Fig 3. Value Distribution: Raw vs Scaled (log-y)')
plt.tight_layout(); plt.savefig(OUT/'fig3_scaling_hist.png', dpi=150); plt.show()

## Step 3 — Temporal Windowing (bonus: temporal dependency)
**Why (Keogh & Lin 2005):** clustering raw subsequences gives meaningless sinusoidal centroids. Fix: aggregate each 60 s window (step 30 s) into `mean + std + delta` per sensor (7×3=21-D vector: level, vibration, trend). Demo below runs on a contiguous head segment (trace + chatter need order); the FINAL windows (Step 4) are built full-file.

#### Cell: Sliding-window features + point-vs-window → `table3_temporal_compare.csv`
**What:** sorts 30k contiguous rows by time, aggregates each 60 s window (step 30 s) into mean + std + delta per sensor (21-D), then compares K-Means on point vs window: S, DBI, chatter-per-second, fit time.
**Why:** mini-P5 with honest reporting: point may win S; window is judged on stability + cost (the temporal bonus). Chatter is normalized per second — raw per-step rates are not comparable.
**Read this:** window chatter/sec must be well below point chatter/sec.

In [ ]:
uw = pd.read_csv(path, usecols=['YYYY/MM/DD','HH:MM:SS']+SENSORS, nrows=N_WIN_SRC, low_memory=True)
for c in SENSORS: uw[c]=pd.to_numeric(uw[c], errors='coerce').astype('float32')
uw = uw.replace(SENTINEL, np.nan).dropna().reset_index(drop=True)
ROP_HI = rop_hi(uw)
assert ROP_HI > 0, 'degenerate sample: ROP all-zero'
uw[ROP] = uw[ROP].clip(upper=ROP_HI)
print('ROP winsorized at HI =', round(ROP_HI, 2))
uw['datetime']=pd.to_datetime(uw['YYYY/MM/DD']+' '+uw['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
uw=uw.sort_values('datetime').reset_index(drop=True)
rows=[]
for st in range(0, len(uw)-W+1, STEP):
    ch=uw.iloc[st:st+W]; rec={'t0':ch['datetime'].iloc[0]}
    for c in SENSORS:
        v=ch[c].to_numpy(dtype='float64')
        rec[c+'__mean']=float(v.mean()); rec[c+'__std']=float(v.std()); rec[c+'__delta']=float(v[-1]-v[0])
    rows.append(rec)
win=pd.DataFrame(rows); feat=[c for c in win.columns if c!='t0']
print('windows:', win.shape, '| features:', len(feat), '(7 sensors x mean/std/delta)')
t0f=time.time(); Xp=RobustScaler().fit_transform(uw[SENSORS].sample(min(15000,len(uw)), random_state=SEED)); sc_p=RobustScaler(); Xp=sc_p.fit_transform(uw[SENSORS].sample(min(15000,len(uw)), random_state=SEED)); km_p=KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Xp); lp=km_p.labels_; tp=round(time.time()-t0f,1)
lp_full=km_p.predict(sc_p.transform(uw[SENSORS].to_numpy()))  # time-ordered full frame: honest chatter
t1f=time.time(); Xw=RobustScaler().fit_transform(win[feat].to_numpy()); lw=KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(Xw); tw=round(time.time()-t1f,1)
sil_p=float(silhouette_score(Xp,lp)); dbi_p=float(davies_bouldin_score(Xp,lp))
sil_w=float(silhouette_score(Xw,lw)); dbi_w=float(davies_bouldin_score(Xw,lw))
chat_p=round(float(np.mean(np.diff(lp_full)!=0)),4); chat_ws=round(float(np.mean(np.diff(lw)!=0)),4); chat_w=round(chat_ws/STEP,5)
print('chatter flips/sec — point:', chat_p, '| window:', chat_w)
pd.DataFrame([{'representation':'point-based','S4848':round(composite(sil_p,dbi_p),4),'silhouette':round(sil_p,4),'dbi':round(dbi_p,4),'chatter_per_step':chat_p,'chatter_per_sec':chat_p,'step_seconds':1,'fit_seconds':tp},{'representation':'window-60s/30s','S4848':round(composite(sil_w,dbi_w),4),'silhouette':round(sil_w,4),'dbi':round(dbi_w,4),'chatter_per_step':chat_ws,'chatter_per_sec':chat_w,'step_seconds':STEP,'fit_seconds':tw}]).to_csv(OUT/'table3_temporal_compare.csv', index=False)
print('Report honestly: point may win S; window wins stability+cost (bonus temporal).')

#### Cell: Temporal trace → `fig4_temporal_trace.png`
**What:** plots the first 3000 seconds of Standpipe Pressure over time.
**Why:** visual evidence that consecutive records are temporally dependent (the assignment's key challenge), motivating the windowing above.
**Read this:** smooth regimes and abrupt jumps = operating-state transitions the clusters should recover.

In [ ]:
plt.figure(figsize=(10,3))
plt.plot(uw['datetime'].iloc[:3000], uw['Standpipe Pressure (psi)'].iloc[:3000], lw=0.7)
plt.title('Fig 4. Standpipe Pressure (first 3000 s) — temporal dependency')
plt.xlabel('Time'); plt.ylabel('psi'); plt.tight_layout()
plt.savefig(OUT/'fig4_temporal_trace.png', dpi=150); plt.show()

## Step 4 — Final Clustering (Elbow + Silhouette + Davies-Bouldin + gate)
**Design:** 4 scenarios = {point, window-60s} x {Standard, Robust}. Points: strided full-span 60k (outage bursts represented). Windows: FULL-FILE chunked gap-aware build (~41k true 60-s windows — never window strided rows, whose spacing is ~12 s). Elbow k=2..8 on 20k subsamples. Then refit best (default `window60-robust`, k=4 → Drilling / Circulation / Connection-Tripping / Idle per Coley 2019) with physics-gate enforcement and centroids in physical units.
**FINAL-LOCK:** ROP winsorized via ROP>0 IQR (`rop_hi`, never 0); Diff Press default OFF (sensitivity via `USE_DIFF_PRESS=True`); S + seed-stability + gate decide; BEST_K follows Fig 5 + domain rule (documented in refit cell).

#### Cell: Final samples + 4 scenarios
**What:** strided full-span 60k point sample AND full-file gap-aware 60-s windows, then the four experiment arms.
**Why:** points cover the whole month (no head bias); windows stay true 60 seconds (strided rows must never be windowed).
**Read this:** printed span (must cover Feb→Mar), window count (~41k), and shapes per scenario.

In [ ]:
usecols=['YYYY/MM/DD','HH:MM:SS']+SENSORS
n_f = 0
for _c in pd.read_csv(path, usecols=[SENSORS[0]], chunksize=500000, low_memory=True):
    n_f += len(_c)
step_f = max(1, n_f // N_FINAL)
fd=pd.read_csv(path, usecols=usecols, skiprows=lambda i: i % step_f != 0, low_memory=True)
print('Final strided sample: every', step_f, 'th row of', n_f, '->', len(fd), 'rows')
for c in SENSORS: fd[c]=pd.to_numeric(fd[c], errors='coerce').astype('float32')
fd=fd.replace(SENTINEL, np.nan)
ROP_HI = rop_hi(fd)
assert ROP_HI > 0, 'degenerate sample: ROP all-zero'
fd[ROP] = fd[ROP].clip(upper=ROP_HI)
print('ROP winsorized at HI =', round(ROP_HI, 2))
print('Modeling sensors:', len(SENSORS), '(Diff Press OFF locked)' if not USE_DIFF_PRESS else '(sensitivity: 8)')
fd=fd.dropna(subset=SENSORS).reset_index(drop=True)
fd['datetime']=pd.to_datetime(fd['YYYY/MM/DD']+' '+fd['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
fd=fd.sort_values('datetime').reset_index(drop=True)
print('final sample:', fd.shape, fd['datetime'].min(), '->', fd['datetime'].max())
def windowed_file(path, w=60, step=30, gap_s=5):
    out=[]
    for chunk in pd.read_csv(path, usecols=['YYYY/MM/DD','HH:MM:SS']+SENSORS, chunksize=200000, low_memory=True):
        for c in SENSORS: chunk[c]=pd.to_numeric(chunk[c], errors='coerce').astype('float32')
        chunk=chunk.replace(SENTINEL, np.nan).dropna(subset=SENSORS).reset_index(drop=True)
        chunk[ROP]=chunk[ROP].clip(upper=ROP_HI)
        chunk['datetime']=pd.to_datetime(chunk['YYYY/MM/DD']+' '+chunk['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
        chunk=chunk.sort_values('datetime').reset_index(drop=True)
        cuts=np.flatnonzero(chunk['datetime'].diff().dt.total_seconds().fillna(0) > gap_s)
        bounds=[0]+(cuts+1).tolist()+[len(chunk)]
        for a,b in zip(bounds[:-1],bounds[1:]):
            seg=chunk.iloc[a:b]
            if len(seg)<w: continue
            for s_ in range(0, len(seg)-w+1, step):
                ch=seg.iloc[s_:s_+w]; rec={'t0':ch['datetime'].iloc[0]}
                for c in SENSORS:
                    v=ch[c].to_numpy(dtype='float64')
                    rec[c+'__mean']=v.mean(); rec[c+'__std']=v.std(); rec[c+'__delta']=v[-1]-v[0]
                out.append(rec)
    return pd.DataFrame(out)
wf=windowed_file(path); feat2=[c for c in wf.columns if c!='t0']
print('full-file windows:', wf.shape, '| features:', len(feat2), '(gap-aware, true 60 s)')
scen={'point-standard':(fd[SENSORS].to_numpy(),StandardScaler()),'point-robust':(fd[SENSORS].to_numpy(),RobustScaler())}
scen['window60-standard']=(wf[feat2].to_numpy(),StandardScaler())
scen['window60-robust']=(wf[feat2].to_numpy(),RobustScaler())
print({k:v[0].shape for k,v in scen.items()})

#### Cell: Elbow loop k=2..8 → `table4_elbow_metrics.csv`
**What:** for each scenario and each k, fits K-Means on a 20k subsample and records inertia, Silhouette, and Davies-Bouldin.
**Why:** Elbow (inertia drop) plus internal metrics jointly select the optimal k without any labels.
**Read this:** the two pivot tables — look for the k where inertia bends and Silhouette peaks (DBI bottoms).

In [ ]:
K_RANGE=range(2,9); rows=[]
for name,(X,sc) in scen.items():
    Xs=sc.fit_transform(X)
    idx=np.random.RandomState(SEED).choice(len(Xs), min(len(Xs),20000), replace=False); Xe=Xs[idx]
    for k in K_RANGE:
        km=KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xe)
        lab=km.labels_
        rows.append({'scenario':name,'k':k,'inertia':float(km.inertia_),'silhouette':float(silhouette_score(Xe,lab)),'dbi':float(davies_bouldin_score(Xe,lab))})
res=pd.DataFrame(rows); res.to_csv(OUT/'table4_elbow_metrics.csv', index=False)
display(res.pivot_table(index='k', columns='scenario', values='silhouette').round(4))
display(res.pivot_table(index='k', columns='scenario', values='dbi').round(4))

#### Cell: Elbow plots → `fig5_elbow_silhouette.png`
**What:** line plots of inertia-vs-k, Silhouette-vs-k, and DBI-vs-k for all four scenarios.
**Why:** the paper's main model-selection figure (Bab III-F); readers pick k visually from the elbow + Sil peak + DBI bottom.
**Read this:** the bend (left), the peak (middle), the bottom (right) — update `BEST_K` in the next cell to match, with the domain rule documented there.

In [ ]:
fig, ax=plt.subplots(1,3, figsize=(15,4))
for name,g in res.groupby('scenario'):
    ax[0].plot(g['k'], g['inertia'], marker='o', label=name)
    ax[1].plot(g['k'], g['silhouette'], marker='o', label=name)
    ax[2].plot(g['k'], g['dbi'], marker='o', label=name)
ax[0].set_title('Fig 5a. Elbow — Inertia vs k'); ax[0].set_xlabel('k')
ax[1].set_title('Fig 5b. Silhouette vs k (higher better)'); ax[1].set_xlabel('k')
ax[2].set_title('Fig 5c. Davies-Bouldin vs k (lower better)'); ax[2].set_xlabel('k')
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/'fig5_elbow_silhouette.png', dpi=150); plt.show()

#### Cell: Best-model refit → `table5_centroids.csv`, `table6_cluster_share.csv`
**What:** refits K-Means with the chosen scenario/k, maps centroids back to physical units, enforces the physics gate, and saves cluster sizes.
**Why:** centroids in psi, klbs, RPM are physically interpretable; map them with Coley (2019) ONLY if the gate passes: WOB>0+RPM>0+ROP>0 = Active Drilling, WOB≈0+ROP=0+pump high = Circulation, pump≈0+RPM≈0 = Connection/Idle.
**Decision rule:** BEST_K default 4 (domain: four rig regimes — elbow curves are flat, documented honestly). If Fig 5 shows a sharp elbow elsewhere, change BEST_K/BEST_SCENARIO and re-run this + next cell. Gate FAIL blocks regime mapping.
**Read this:** each centroid row must read like a real rig state — that mapping is the Bab III-F discussion.

In [ ]:
BEST_SCENARIO='window60-robust'; BEST_K=4  # domain rule (flat elbow) — change after reading Fig 5
Xb, scb = scen[BEST_SCENARIO]
Xbs=scb.fit_transform(Xb)
t0f=time.time(); kmf=KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Xbs); print('refit seconds:', round(time.time()-t0f,1))
labf=kmf.labels_
cent=scb.inverse_transform(kmf.cluster_centers_)
if 'window' in BEST_SCENARIO:
    mc=[c for c in feat2 if c.endswith('__mean')]
    mi=[feat2.index(c) for c in mc]  # positional: 21-D features interleave mean/std/delta
    ctab=pd.DataFrame(cent[:,mi], columns=[c.replace('__mean','') for c in mc])
else:
    ctab=pd.DataFrame(cent, columns=SENSORS)
ctab.index.name='cluster'; display(ctab.round(2))
passed, fails = physicality_gate(ctab)
print('physics gate:', 'PASS' if passed else 'FAIL -> ' + ';'.join(fails))
ctab.round(2).to_csv(OUT/'table5_centroids.csv')
pd.Series(labf).value_counts(normalize=True).round(4).to_csv(OUT/'table6_cluster_share.csv')
if passed:
    print('Map with Coley (2019): WOB>0+RPM>0+ROP>0 => Active Drilling; WOB~0+ROP=0+pump high => Circulation & Hole Cleaning; pump~0+RPM~0 => Pipe Connection; else Idle.')
else:
    print('GATE FAILED — do NOT map regimes; fix preprocessing/sample, then re-run from the final-sample cell.')

#### Cell: PCA scatter → `fig6_pca_clusters.png`
**What:** projects 15k points to 2-D with PCA and colors them by cluster.
**Why:** visual sanity check of cluster separation for the paper; compact, separated blobs = good clustering.
**Read this:** overlapping clouds suggest trying a different k or the window representation.

In [ ]:
p2=PCA(n_components=2, random_state=SEED).fit_transform(Xbs[np.random.RandomState(SEED).choice(len(Xbs), min(len(Xbs),15000), replace=False)])
ls=labf[np.random.RandomState(SEED).choice(len(labf), p2.shape[0], replace=False)]
plt.figure(figsize=(6,5)); plt.scatter(p2[:,0], p2[:,1], c=ls, s=4, cmap='tab10', alpha=0.6)
plt.title(f'Fig 6. PCA 2D — {BEST_SCENARIO} k={BEST_K}')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout()
plt.savefig(OUT/'fig6_pca_clusters.png', dpi=150); plt.show()

## Step 5 — Export checklist for IEEE paper
- Kaggle: download `/kaggle/output/`; Colab: outputs in `fathan_clustering_outputs/` (zip auto-downloads below). Files: `table0–table6`, `fig1–fig6`.
- Tables → Bab II/III; Figures → same numbers (already English captions).
- Appendix: Kaggle Dataset link + this Notebook link + GitHub repo link.
- Add Gen-AI declaration (required by assignment).
- If Elbow suggests k≠4, update `BEST_K` and re-run only the last 2 cells (refit + PCA).

## Recap — verdicts + MISSING-check (read before writing Bab III-F)
- P1 scaler: winner = highest S among gate-passed (expect robust).
- P5 demo: honest S (point may win) + window must win chatter/sec.
- Elbow: BEST_K follows Fig 5 + domain rule (default 4); DBI bottom must agree with Sil peak.
- Refit: gate must PASS before any regime mapping.
- Run the next cell: it inventories all 13 paper artifacts and prints `MISSING: none` when safe to write.

In [ ]:
import pandas as pd
from pathlib import Path
need = ['table0_sensor_taxonomy.csv','table1_descriptive.csv','fig1_correlation.png','fig2_boxplots.png','table2_scaler_compare.csv','fig3_scaling_hist.png','table3_temporal_compare.csv','fig4_temporal_trace.png','table4_elbow_metrics.csv','fig5_elbow_silhouette.png','table5_centroids.csv','table6_cluster_share.csv','fig6_pca_clusters.png']
missing=[f for f in need if not (OUT/f).exists()]
print('MISSING:', missing if missing else 'none — all 13 paper artifacts present')
t2=pd.read_csv(OUT/'table2_scaler_compare.csv', index_col=0)
print('P1 winner (highest S, gate-passed):', t2['S4848'].astype(float).idxmax())
t3=pd.read_csv(OUT/'table3_temporal_compare.csv')
print('P5 demo S winner:', t3.sort_values('S4848',ascending=False).iloc[0]['representation'])
t4=pd.read_csv(OUT/'table4_elbow_metrics.csv')
print(t4.pivot_table(index='k',columns='scenario',values='silhouette').round(4).to_string())

In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT files:', moved if moved else '<EMPTY — cells above never ran>')
print('OUT dir (absolute):', OUT.resolve())
zp = live / 'fathan_clustering_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for n in moved:
        z.write(live / n, n)
print('packed:', zp.resolve())
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zp))
        print('browser download started.')
    except Exception as e:
        print('auto-download failed — grab the zip from the Files pane:', live.resolve(), e)